# 港口目录逐项核查

粒度为已发布目录 ID，不把港区、泊位和海上终端当作互不重叠的港口。代码只读本地快照，默认不联网、不采集、不写库。水域修正采用具名主要海域，避免把监测矩形当作实际水域。

参照是 DOE NETL 的 NGA WPI 镜像，编辑时间不是逐港核实时间；其完整性也不能代表世界所有港口。104 条未对应项需要进一步身份消歧。本次使用的来源、日期、原文摘录、例外和字段修正均在同目录 JSON/CSV 中。


In [1]:
import json, hashlib
from pathlib import Path
from collections import Counter
import duckdb
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == "audits": ROOT = ROOT.parent
assert (ROOT / "port_metadata.py").exists()
audit = json.loads((ROOT / "audits/port_catalog_review_2026-10-16.json").read_text())
reference_path = ROOT / "audits/port_catalog_reference_wpi_2026-10-16.json"
assert hashlib.sha256(reference_path.read_bytes()).hexdigest() == audit["baseline_receipt"]["sha256"]
assert hashlib.sha256((ROOT / "audits/port_type_review_2026-10-16.json").read_bytes()).hexdigest() == audit["original_business_review_sha256"]
reference = json.loads(reference_path.read_text())
assert not reference.get("exceededTransferLimit", False)
print("Reference entries:", len(reference["features"]))

Reference entries: 204


In [2]:
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from port_metadata import apply_port_metadata
from port_types import port_type_label, port_business_label
with duckdb.connect(str(ROOT / "runtime/monitoring.duckdb"), read_only=True) as conn:
    raw = [json.loads(row[0]) for row in conn.execute("SELECT payload FROM catalogs WHERE kind='ports'").fetchall()]
current = {p["portid"]: p for p in apply_port_metadata(raw)}
assert len(current) == len(raw) == audit["catalog_size"] == 132
for checked in audit["ports"]:
    p = current[checked["portid"]]
    assert p["name"] == checked["name_source"]
    for field in ["name_cn", "country", "region", "lat", "lon"]:
        assert p[field] == checked[field], (p["portid"], field)
    assert port_type_label(p) == checked["category"]
    assert port_business_label(p) == checked["business_description"]
print("All 132 IDs reconciled; corrections are a read-only projection or persisted by the sole publisher.")
print("Countries:", dict(Counter(p["country"] for p in current.values())))
print("Water labels:", dict(Counter(p["region"] for p in current.values())))
print("Corrections:", audit["correction_counts"])

All 132 IDs reconciled; corrections are a read-only projection or persisted by the sole publisher.
Countries: {'United Arab Emirates': 34, 'Bahrain': 5, 'Qatar': 8, 'Iraq': 7, 'Oman': 10, 'Saudi Arabia': 24, 'Iran': 21, 'Yemen': 5, 'Egypt': 9, 'Kuwait': 6, 'Djibouti': 2, 'Eritrea': 1}
Water labels: {'波斯湾': 81, '阿曼湾': 10, '红海': 19, '阿拉伯海': 3, '霍尔木兹海峡': 4, '地中海': 2, '亚丁湾': 3, '苏伊士湾': 6, '里海': 3, '苏伊士运河': 1}
Corrections: {'water_labels': 32, 'chinese_names': 2, 'additional_WPI_crosswalks': 9}


In [3]:
reference_ids = {int(p["attributes"]["wpinumber"]) for p in reference["features"]}
matched = {i for p in audit["ports"] for i in p["wpi_ids"]} & reference_ids
missing = pd.DataFrame(audit["unmatched_reference_entries"])
assert len(matched) == audit["matched_baseline_wpi_ids"] == 100
assert len(missing) == len(reference_ids - matched) == 104
print("Matched WPI IDs:", len(matched))
print("Unmatched reference entries (not a count of non-overlapping missing ports):", len(missing))
print(missing.groupby("country").size().to_string())
print("Limitations:")
for note in audit["scope_limitations"]: print("-", note)

Matched WPI IDs: 100
Unmatched reference entries (not a count of non-overlapping missing ports): 104
country
Cyprus      6
Egypt      21
Eritrea     1
Israel      5
Jordan      1
Lebanon     4
Sudan       4
Syria       3
Turkey     54
Yemen       5
Limitations:
- Not all Middle East ports are included. Existing acquisition selects five monitoring boxes plus eight Gulf-producer countries.
- The reference covers major ports/terminals only. It includes overlapping facility levels and is not an exhaustive list of all private jetties or fishing harbors.
- No Palestine/Gaza matches in the comparator query; this is not evidence of no ports.
- 104 unmatched reference entries need further port-vs-terminal identity resolution before importing or assigning traffic.


In [4]:
checked = pd.DataFrame(audit["ports"])
columns = ["portid", "name_cn", "name_source", "country", "region", "category", "business_description", "exceptions"]
print("Per-ID evidence is in audits/port_catalog_review_2026-10-16.json and the companion CSV.")
print(checked.loc[checked["exceptions"].map(bool), columns].to_string(index=False))
print("No upstream collection, database mutation, or new task was executed by this notebook.")

Per-ID evidence is in audits/port_catalog_review_2026-10-16.json and the companion CSV.
  portid         name_cn                            name_source              country region    category business_description                                                                  exceptions
  fso151 阿曼海上石油终端1（源库名称）         Oman - Offshore Oil Terminal 1                 Oman   阿拉伯海      油品港及终端       海上油品终端（源目录未具名）                                               [来源未具名；无法将占位名称认定为已确认的设施正式名称。]
  fso158         海上石油终端1 Saudi Arabia - Offshore Oil Terminal 1         Saudi Arabia    波斯湾      油品港及终端       海上油品终端（源目录未具名）                                               [来源未具名；无法将占位名称认定为已确认的设施正式名称。]
wpi48035       苏赫纳港气体船码头        Sokhna Port Gas Tanker Terminal                Egypt   苏伊士湾         待核实              气体船专用码头                                                [气体船专用码头，既有证据未区分 LNG 与 LPG。]
wpi48268        穆巴拉兹石油码头                  Mubarraz Oil Terminal United Arab Emirates    波斯湾      油品港及终端     